# Healthcare Facilities Data Cleaning Pipeline
## Raw Excel → raw_facilities → clean_facilities

This notebook transforms raw Indian healthcare facility data into a clean, queryable Delta table in Unity Catalog.

**Input:** `/Volumes/workspace/default/hackathon/VF_Hackathon_Dataset_India_Large.xlsx`  
**Output:** `workspace.default.clean_facilities` (Unity Catalog table)

### Pipeline Steps:
0. Load Excel from Volume into Delta
1. Clean null values ("null", "[]")
2. Fix state field mappings (Kochi→Kerala, Bengaluru→Karnataka)
3. Parse JSON arrays (specialties)
4. Fix typo: farmacy → pharmacy
5. Cast numeric columns to integers
6. Extract clean 10-digit phone numbers
7. Add boolean flags (has_emergency, has_icu, has_obg)
8. Build full_text_blob for vector search
9. Add stable facility_id (MD5 hash)
10. Write to Unity Catalog

In [0]:
# Step 0: Load Excel from Volume into raw Delta table
from pyspark.sql import SparkSession
import pyspark.sql.functions as F

# Path to Excel file in Volumes
excel_path = "/Volumes/workspace/default/hackathon/VF_Hackathon_Dataset_India_Large.xlsx"

# Load Excel file using Databricks built-in reader
raw_df = spark.read.format("excel") \
    .option("header", "true") \
    .option("inferSchema", "true") \
    .load(excel_path)

# Write to raw_facilities Delta table
raw_df.write.format("delta").mode("overwrite").saveAsTable("workspace.default.raw_facilities")

print(f"✓ Loaded {raw_df.count()} rows into raw_facilities")
raw_df.printSchema()
display(raw_df.limit(5))

✓ Loaded 10001 rows into raw_facilities
root
 |-- _c0: string (nullable = true)
 |-- _c1: string (nullable = true)
 |-- _c2: string (nullable = true)
 |-- _c3: string (nullable = true)
 |-- _c4: string (nullable = true)
 |-- _c5: string (nullable = true)
 |-- _c6: string (nullable = true)
 |-- _c7: string (nullable = true)
 |-- _c8: string (nullable = true)
 |-- _c9: string (nullable = true)
 |-- _c10: string (nullable = true)
 |-- _c11: string (nullable = true)
 |-- _c12: string (nullable = true)
 |-- _c13: string (nullable = true)
 |-- _c14: string (nullable = true)
 |-- _c15: string (nullable = true)
 |-- _c16: string (nullable = true)
 |-- _c17: string (nullable = true)
 |-- _c18: string (nullable = true)
 |-- _c19: string (nullable = true)
 |-- _c20: string (nullable = true)
 |-- _c21: string (nullable = true)
 |-- _c22: string (nullable = true)
 |-- _c23: string (nullable = true)
 |-- _c24: string (nullable = true)
 |-- _c25: string (nullable = true)
 |-- _c26: string (nullable =

_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9,_c10,_c11,_c12,_c13,_c14,_c15,_c16,_c17,_c18,_c19,_c20,_c21,_c22,_c23,_c24,_c25,_c26,_c27,_c28,_c29,_c30,_c31,_c32,_c33,_c34,_c35,_c36,_c37,_c38,_c39,_c40
name,phone_numbers,officialPhone,email,websites,officialWebsite,yearEstablished,facebookLink,twitterLink,linkedinLink,instagramLink,address_line1,address_line2,address_line3,address_city,address_stateOrRegion,address_zipOrPostcode,address_country,address_countryCode,facilityTypeId,operatorTypeId,affiliationTypeIds,description,numberDoctors,capacity,specialties,procedure,equipment,capability,recency_of_page_update,distinct_social_media_presence_count,affiliated_staff_presence,custom_logo_presence,number_of_facts_about_the_organization,post_metrics_most_recent_social_media_post_date,post_metrics_post_count,engagement_metrics_n_followers,engagement_metrics_n_likes,engagement_metrics_n_engagements,latitude,longitude
1000 Smiles Dental Clinic,"[""+919392803399""]",+919392803399,1000smilesdentalclinic@gmail.com,"[""https://www.facebook.com/249768584883702"",""https://www.justdial.com/Hyderabad/1000-Smiles-Dental-Clinic-Opposie-Hdfc-Bank-Amberpet/040PXX40-XX40-160227012741-N9P1_BZDET""]",null,null,https://www.facebook.com/249768584883702,null,null,null,"Plot No 2-2-1075/15/1/5, Shivam Road","Opposite Hdfc Bank, Tilak Nagar","Near Old Mla Quarters, Hyderguda",Hyderabad,Telangana,500013.0,India,IN,clinic,null,null,"Dental clinic offering RCT (Root Canal) and Laser Dentistry in Amberpet, Hyderabad.",null,null,"[""familyMedicine"",""periodontics"",""endodontics"",""dentistry"",""aestheticDentistry""]","[""Performs root canal therapy (RCT)"",""Provides laser dentistry services""]",[],"[""Has been in operation for 10 years"",""Dental clinic""]",null,1.0,false,false,null,null,null,23.0,10.0,null,17.3977394104003,78.482681274414
108 Eye And Heath Centre,"[""+919015821616""]",+919015821616,108healthcentre@gmail.com,"[""https://www.indiamart.com/108-eye-and-health-centre/"",""https://108-eye-health.localo.site/"",""108eye.com"",""https://www.justdial.com/Noida/108-Eye-And-Heath-Centre-Near-State-Bank-Of-India-D-Block-Sector-50-Noida-Sector-50/011PXX11-XX11-180216202406-Z2S8_BZDET"",""https://www.facebook.com/108eyeandhealthcentre/"",""https://108eye.com/"",""https://www.practo.com/noida/clinics/eye-clinics"",""https://www.facebook.com/252399348484757""]",108eye.com,null,https://www.facebook.com/252399348484757,null,null,null,House Number 108,"Near State Bank Of India D Block Sector 50, E Block",null,Noida,Uttar Pradesh,201307.0,India,IN,clinic,private,null,"108 Eye And Health Centre provides you the best range of treatments, treatment services, pharmaceutical injection, veterinary drugs & our services with effective & timely delivery.",1.0,null,"[""pediatricsAndStrabismusOphthalmology"",""ophthalmology"",""refractiveSurgeryOphthalmology"",""familyMedicine"",""eyeTraumaAndEmergencyEyeCare"",""cataractAndAnteriorSegmentSurgery"",""uveitisOphthalmology"",""retinaAndVitreoretinalOphthalmology"",""glaucomaOphthalmology"",""neuroOphthalmology""]","[""Treats Squint"",""Glaucoma management"",""Offers Paediatric Ophthalmology"",""Performs cataract surgery"",""General eye check-up"",""Manages Uveitis"",""Cataract surgery"",""LASIK"",""IPCL"",""Pre-LASIK tests"",""Provides Basic Eye Check Up"",""Offers Retina Treatment Services"",""Offers Glaucoma Treatment"",""Offers Neuro Ophthalmology"",""ICL"",""Post-LASIK care"",""Squint correction"",""Retina treatment"",""Pediatric ophthalmology services""]",[],"[""Neuro Ophthalmology services offered"",""Outpatient ophthalmology clinic"",""Has 1 ophthalmologist/eye surgeon on staff"",""Dr. B K Varma is an ophthalmologist/eye surgeon with 17 years of experience"",""Retina Treatment Services offered"",""Paediatric Ophthalmology services offered"",""Employs experienced ophthalmologists"",""Uses latest equipment and world-standard technologies in eye care"",""Provides comprehensive eye care services under one roof""]",null,3.0

In [0]:
# Step 1: Replace string "null" and "[]" with actual null
from pyspark.sql.functions import when, col, trim

df = spark.table("workspace.default.raw_facilities")

# Get all string columns
string_cols = [field.name for field in df.schema.fields if str(field.dataType) == "StringType"]

# Replace "null", "NULL", "[]" with None
for col_name in string_cols:
    df = df.withColumn(
        col_name,
        when(
            (trim(col(col_name)).isin("null", "NULL", "[]", "", "None")),
            None
        ).otherwise(col(col_name))
    )

print(f"✓ Cleaned {len(string_cols)} string columns")
display(df.limit(5))

✓ Cleaned 0 string columns


_c0,_c1,_c2,_c3,_c4,_c5,_c6,_c7,_c8,_c9,_c10,_c11,_c12,_c13,_c14,_c15,_c16,_c17,_c18,_c19,_c20,_c21,_c22,_c23,_c24,_c25,_c26,_c27,_c28,_c29,_c30,_c31,_c32,_c33,_c34,_c35,_c36,_c37,_c38,_c39,_c40
name,phone_numbers,officialPhone,email,websites,officialWebsite,yearEstablished,facebookLink,twitterLink,linkedinLink,instagramLink,address_line1,address_line2,address_line3,address_city,address_stateOrRegion,address_zipOrPostcode,address_country,address_countryCode,facilityTypeId,operatorTypeId,affiliationTypeIds,description,numberDoctors,capacity,specialties,procedure,equipment,capability,recency_of_page_update,distinct_social_media_presence_count,affiliated_staff_presence,custom_logo_presence,number_of_facts_about_the_organization,post_metrics_most_recent_social_media_post_date,post_metrics_post_count,engagement_metrics_n_followers,engagement_metrics_n_likes,engagement_metrics_n_engagements,latitude,longitude
1000 Smiles Dental Clinic,"[""+919392803399""]",+919392803399,1000smilesdentalclinic@gmail.com,"[""https://www.facebook.com/249768584883702"",""https://www.justdial.com/Hyderabad/1000-Smiles-Dental-Clinic-Opposie-Hdfc-Bank-Amberpet/040PXX40-XX40-160227012741-N9P1_BZDET""]",null,null,https://www.facebook.com/249768584883702,null,null,null,"Plot No 2-2-1075/15/1/5, Shivam Road","Opposite Hdfc Bank, Tilak Nagar","Near Old Mla Quarters, Hyderguda",Hyderabad,Telangana,500013.0,India,IN,clinic,null,null,"Dental clinic offering RCT (Root Canal) and Laser Dentistry in Amberpet, Hyderabad.",null,null,"[""familyMedicine"",""periodontics"",""endodontics"",""dentistry"",""aestheticDentistry""]","[""Performs root canal therapy (RCT)"",""Provides laser dentistry services""]",[],"[""Has been in operation for 10 years"",""Dental clinic""]",null,1.0,false,false,null,null,null,23.0,10.0,null,17.3977394104003,78.482681274414
108 Eye And Heath Centre,"[""+919015821616""]",+919015821616,108healthcentre@gmail.com,"[""https://www.indiamart.com/108-eye-and-health-centre/"",""https://108-eye-health.localo.site/"",""108eye.com"",""https://www.justdial.com/Noida/108-Eye-And-Heath-Centre-Near-State-Bank-Of-India-D-Block-Sector-50-Noida-Sector-50/011PXX11-XX11-180216202406-Z2S8_BZDET"",""https://www.facebook.com/108eyeandhealthcentre/"",""https://108eye.com/"",""https://www.practo.com/noida/clinics/eye-clinics"",""https://www.facebook.com/252399348484757""]",108eye.com,null,https://www.facebook.com/252399348484757,null,null,null,House Number 108,"Near State Bank Of India D Block Sector 50, E Block",null,Noida,Uttar Pradesh,201307.0,India,IN,clinic,private,null,"108 Eye And Health Centre provides you the best range of treatments, treatment services, pharmaceutical injection, veterinary drugs & our services with effective & timely delivery.",1.0,null,"[""pediatricsAndStrabismusOphthalmology"",""ophthalmology"",""refractiveSurgeryOphthalmology"",""familyMedicine"",""eyeTraumaAndEmergencyEyeCare"",""cataractAndAnteriorSegmentSurgery"",""uveitisOphthalmology"",""retinaAndVitreoretinalOphthalmology"",""glaucomaOphthalmology"",""neuroOphthalmology""]","[""Treats Squint"",""Glaucoma management"",""Offers Paediatric Ophthalmology"",""Performs cataract surgery"",""General eye check-up"",""Manages Uveitis"",""Cataract surgery"",""LASIK"",""IPCL"",""Pre-LASIK tests"",""Provides Basic Eye Check Up"",""Offers Retina Treatment Services"",""Offers Glaucoma Treatment"",""Offers Neuro Ophthalmology"",""ICL"",""Post-LASIK care"",""Squint correction"",""Retina treatment"",""Pediatric ophthalmology services""]",[],"[""Neuro Ophthalmology services offered"",""Outpatient ophthalmology clinic"",""Has 1 ophthalmologist/eye surgeon on staff"",""Dr. B K Varma is an ophthalmologist/eye surgeon with 17 years of experience"",""Retina Treatment Services offered"",""Paediatric Ophthalmology services offered"",""Employs experienced ophthalmologists"",""Uses latest equipment and world-standard technologies in eye care"",""Provides comprehensive eye care services under one roof""]",null,3.0

In [0]:
# Fix: Extract actual column names from first row
from pyspark.sql.functions import col

# Get the first row which contains the actual column names
first_row = df.first()
actual_column_names = [first_row[i] for i in range(len(df.columns))]

print(f"Extracted column names: {actual_column_names[:5]}...")  # Show first 5

# Rename columns using the actual names from row 1
for old_col, new_col in zip(df.columns, actual_column_names):
    df = df.withColumnRenamed(old_col, new_col)

# Remove the first row (which was the header row)
df = df.filter(col("name") != "name")  # Filter out the header row itself

print(f"✓ Fixed column names - now have proper headers")
print(f"✓ Removed header row from data")
display(df.limit(5))

Extracted column names: ['name', 'phone_numbers', 'officialPhone', 'email', 'websites']...
✓ Fixed column names - now have proper headers
✓ Removed header row from data


name,phone_numbers,officialPhone,email,websites,officialWebsite,yearEstablished,facebookLink,twitterLink,linkedinLink,instagramLink,address_line1,address_line2,address_line3,address_city,address_stateOrRegion,address_zipOrPostcode,address_country,address_countryCode,facilityTypeId,operatorTypeId,affiliationTypeIds,description,numberDoctors,capacity,specialties,procedure,equipment,capability,recency_of_page_update,distinct_social_media_presence_count,affiliated_staff_presence,custom_logo_presence,number_of_facts_about_the_organization,post_metrics_most_recent_social_media_post_date,post_metrics_post_count,engagement_metrics_n_followers,engagement_metrics_n_likes,engagement_metrics_n_engagements,latitude,longitude
1000 Smiles Dental Clinic,"[""+919392803399""]",+919392803399,1000smilesdentalclinic@gmail.com,"[""https://www.facebook.com/249768584883702"",""https://www.justdial.com/Hyderabad/1000-Smiles-Dental-Clinic-Opposie-Hdfc-Bank-Amberpet/040PXX40-XX40-160227012741-N9P1_BZDET""]",null,null,https://www.facebook.com/249768584883702,null,null,null,"Plot No 2-2-1075/15/1/5, Shivam Road","Opposite Hdfc Bank, Tilak Nagar","Near Old Mla Quarters, Hyderguda",Hyderabad,Telangana,500013.0,India,IN,clinic,null,null,"Dental clinic offering RCT (Root Canal) and Laser Dentistry in Amberpet, Hyderabad.",null,null,"[""familyMedicine"",""periodontics"",""endodontics"",""dentistry"",""aestheticDentistry""]","[""Performs root canal therapy (RCT)"",""Provides laser dentistry services""]",[],"[""Has been in operation for 10 years"",""Dental clinic""]",null,1.0,false,false,null,null,null,23.0,10.0,null,17.3977394104003,78.482681274414
108 Eye And Heath Centre,"[""+919015821616""]",+919015821616,108healthcentre@gmail.com,"[""https://www.indiamart.com/108-eye-and-health-centre/"",""https://108-eye-health.localo.site/"",""108eye.com"",""https://www.justdial.com/Noida/108-Eye-And-Heath-Centre-Near-State-Bank-Of-India-D-Block-Sector-50-Noida-Sector-50/011PXX11-XX11-180216202406-Z2S8_BZDET"",""https://www.facebook.com/108eyeandhealthcentre/"",""https://108eye.com/"",""https://www.practo.com/noida/clinics/eye-clinics"",""https://www.facebook.com/252399348484757""]",108eye.com,null,https://www.facebook.com/252399348484757,null,null,null,House Number 108,"Near State Bank Of India D Block Sector 50, E Block",null,Noida,Uttar Pradesh,201307.0,India,IN,clinic,private,null,"108 Eye And Health Centre provides you the best range of treatments, treatment services, pharmaceutical injection, veterinary drugs & our services with effective & timely delivery.",1.0,null,"[""pediatricsAndStrabismusOphthalmology"",""ophthalmology"",""refractiveSurgeryOphthalmology"",""familyMedicine"",""eyeTraumaAndEmergencyEyeCare"",""cataractAndAnteriorSegmentSurgery"",""uveitisOphthalmology"",""retinaAndVitreoretinalOphthalmology"",""glaucomaOphthalmology"",""neuroOphthalmology""]","[""Treats Squint"",""Glaucoma management"",""Offers Paediatric Ophthalmology"",""Performs cataract surgery"",""General eye check-up"",""Manages Uveitis"",""Cataract surgery"",""LASIK"",""IPCL"",""Pre-LASIK tests"",""Provides Basic Eye Check Up"",""Offers Retina Treatment Services"",""Offers Glaucoma Treatment"",""Offers Neuro Ophthalmology"",""ICL"",""Post-LASIK care"",""Squint correction"",""Retina treatment"",""Pediatric ophthalmology services""]",[],"[""Neuro Ophthalmology services offered"",""Outpatient ophthalmology clinic"",""Has 1 ophthalmologist/eye surgeon on staff"",""Dr. B K Varma is an ophthalmologist/eye surgeon with 17 years of experience"",""Retina Treatment Services offered"",""Paediatric Ophthalmology services offered"",""Employs experienced ophthalmologists"",""Uses latest equipment and world-standard technologies in eye care"",""Provides comprehensive eye care services under one roof""]",null,3.0,true,true,null,null,null,null,null,null,28.57222366333,77.3690338134765
14 Stars Dental,"[""+918791556586""]",+918791556586,doctor@14starsdental.com,"[""14starsdental.com"",""https://www.faceboo

In [0]:
# Step 2: Fix state field - map cities to proper states
from pyspark.sql.functions import when, col

# City to state mapping (30 bad values)
city_to_state_mapping = {
    "Kochi": "Kerala",
    "Bengaluru": "Karnataka",
    "Bangalore": "Karnataka",
    "Mumbai": "Maharashtra",
    "Pune": "Maharashtra",
    "Delhi": "Delhi",
    "Hyderabad": "Telangana",
    "Chennai": "Tamil Nadu",
    "Kolkata": "West Bengal",
    "Ahmedabad": "Gujarat",
    "Surat": "Gujarat",
    "Jaipur": "Rajasthan",
    "Lucknow": "Uttar Pradesh",
    "Kanpur": "Uttar Pradesh",
    "Nagpur": "Maharashtra",
    "Indore": "Madhya Pradesh",
    "Bhopal": "Madhya Pradesh",
    "Visakhapatnam": "Andhra Pradesh",
    "Patna": "Bihar",
    "Vadodara": "Gujarat",
    "Ghaziabad": "Uttar Pradesh",
    "Ludhiana": "Punjab",
    "Agra": "Uttar Pradesh",
    "Nashik": "Maharashtra",
    "Faridabad": "Haryana",
    "Meerut": "Uttar Pradesh",
    "Rajkot": "Gujarat",
    "Varanasi": "Uttar Pradesh",
    "Srinagar": "Jammu and Kashmir",
    "Amritsar": "Punjab"
}

# Apply mapping - actual column name is address_stateOrRegion
if "address_stateOrRegion" in df.columns:
    state_case = col("address_stateOrRegion")
    for city, state in city_to_state_mapping.items():
        state_case = when(col("address_stateOrRegion") == city, state).otherwise(state_case)
    df = df.withColumn("address_stateOrRegion", state_case)
    print("✓ Fixed state mappings")
else:
    print("⚠ No 'address_stateOrRegion' column found")

display(df.select("address_stateOrRegion", "address_city").distinct().limit(20))

✓ Fixed state mappings


address_stateOrRegion,address_city
Telangana,Hyderabad
Uttar Pradesh,Noida
Uttar Pradesh,Aurangabad
Uttar Pradesh,Ghaziabad
Gujarat,Ahmedabad
Maharashtra,Pune
Punjab,Dinanagar
Punjab,Amritsar
Tamil Nadu,Chidambaram
Tamil Nadu,Tiruppur


In [0]:
# Step 3: Parse JSON arrays - convert string arrays to actual arrays
from pyspark.sql.functions import from_json, col, when
from pyspark.sql.types import ArrayType, StringType

# Columns that contain JSON arrays (actual column names from the data)
json_array_columns = ["specialties", "procedure", "equipment", "capability", "phone_numbers", "websites"]

for col_name in json_array_columns:
    if col_name in df.columns:
        # Get the current column's data type from the schema
        field = [f for f in df.schema.fields if f.name == col_name][0]
        col_type = type(field.dataType).__name__
        
        # Only parse if it's a StringType (not already an ArrayType)
        if col_type == "StringType":
            # Parse JSON string to array
            df = df.withColumn(
                col_name,
                when(
                    col(col_name).isNotNull(),
                    from_json(col(col_name), ArrayType(StringType()))
                ).otherwise(None)
            )
            print(f"✓ Parsed {col_name} from string to array")
        elif col_type == "ArrayType":
            print(f"✓ {col_name} is already an array - skipping")
        else:
            print(f"⚠ {col_name} has unexpected type: {col_type}")

display(df.select("specialties", "procedure", "equipment").limit(5))

✓ Parsed specialties from string to array
✓ Parsed procedure from string to array
✓ Parsed equipment from string to array
✓ Parsed capability from string to array
✓ Parsed phone_numbers from string to array
✓ Parsed websites from string to array


specialties,procedure,equipment
"List(familyMedicine, periodontics, endodontics, dentistry, aestheticDentistry)","List(Performs root canal therapy (RCT), Provides laser dentistry services)",List()
"List(pediatricsAndStrabismusOphthalmology, ophthalmology, refractiveSurgeryOphthalmology, familyMedicine, eyeTraumaAndEmergencyEyeCare, cataractAndAnteriorSegmentSurgery, uveitisOphthalmology, retinaAndVitreoretinalOphthalmology, glaucomaOphthalmology, neuroOphthalmology)","List(Treats Squint, Glaucoma management, Offers Paediatric Ophthalmology, Performs cataract surgery, General eye check-up, Manages Uveitis, Cataract surgery, LASIK, IPCL, Pre-LASIK tests, Provides Basic Eye Check Up, Offers Retina Treatment Services, Offers Glaucoma Treatment, Offers Neuro Ophthalmology, ICL, Post-LASIK care, Squint correction, Retina treatment, Pediatric ophthalmology services)",List()
"List(dentistry, generalDentistry)",List(),List()
List(familyMedicine),List(),List()
List(familyMedicine),null,null


In [0]:
# Step 4: Fix typo - "farmacy" -> "pharmacy" in facilityTypeId
from pyspark.sql.functions import regexp_replace, col

# Check if facilityTypeId column exists
if "facilityTypeId" in df.columns:
    df = df.withColumn(
        "facilityTypeId",
        regexp_replace(col("facilityTypeId"), "farmacy", "pharmacy")
    )
    df = df.withColumn(
        "facilityTypeId",
        regexp_replace(col("facilityTypeId"), "Farmacy", "Pharmacy")
    )
    print("✓ Fixed farmacy → pharmacy typo")
    display(df.select("facilityTypeId").distinct())
else:
    print("⚠ No 'facilityTypeId' column found")

✓ Fixed farmacy → pharmacy typo


facilityTypeId
clinic
dentist
hospital
pharmacy
doctor


In [0]:
# Step 5: Cast numeric columns to integers using try_cast
# try_cast converts malformed values (like string "null") to NULL instead of failing

# Numeric columns to cast (actual column names from the data)
numeric_columns = [
    "numberDoctors", "capacity",
    "distinct_social_media_presence_count",
    "number_of_facts_about_the_organization",
    "post_metrics_post_count",
    "engagement_metrics_n_followers",
    "engagement_metrics_n_likes",
    "engagement_metrics_n_engagements"
]

# Build selectExpr to use try_cast for numeric columns
select_exprs = []
for field in df.schema.fields:
    if field.name in numeric_columns:
        # Use try_cast to handle string "null" and other malformed values
        select_exprs.append(f"try_cast({field.name} as int) as {field.name}")
        print(f"✓ Will cast {field.name} to integer with try_cast")
    else:
        select_exprs.append(field.name)

# Apply the transformation
df = df.selectExpr(*select_exprs)

print("\n✓ Numeric columns cast complete (malformed values converted to NULL)")

✓ Will cast numberDoctors to integer with try_cast
✓ Will cast capacity to integer with try_cast
✓ Will cast distinct_social_media_presence_count to integer with try_cast
✓ Will cast number_of_facts_about_the_organization to integer with try_cast
✓ Will cast post_metrics_post_count to integer with try_cast
✓ Will cast engagement_metrics_n_followers to integer with try_cast
✓ Will cast engagement_metrics_n_likes to integer with try_cast
✓ Will cast engagement_metrics_n_engagements to integer with try_cast

✓ Numeric columns cast complete (malformed values converted to NULL)


In [0]:
# Step 6: Extract clean 10-digit phone number from officialPhone
# Using SQL expression to avoid namespace issues

if "officialPhone" in df.columns:
    # Drop clean_phone if it already exists to avoid ambiguous reference
    if "clean_phone" in df.columns:
        df = df.drop("clean_phone")
    
    # Use SQL expression for the transformation
    df = df.selectExpr(
        "*",
        """CASE 
            WHEN length(regexp_replace(officialPhone, '[^0-9]', '')) = 10 
            THEN regexp_replace(officialPhone, '[^0-9]', '')
            WHEN length(regexp_replace(officialPhone, '[^0-9]', '')) > 10 
                AND substring(regexp_replace(officialPhone, '[^0-9]', ''), 1, 2) = '91'
            THEN substring(regexp_replace(officialPhone, '[^0-9]', ''), 3, 10)
            ELSE NULL
        END as clean_phone"""
    )
    
    print("✓ Extracted clean 10-digit phone numbers")
    display(df.select("officialPhone", "clean_phone").limit(10))
else:
    print("⚠ No phone column found")

✓ Extracted clean 10-digit phone numbers


officialPhone,clean_phone
+919392803399,9392803399
+919015821616,9015821616
+918791556586,8791556586
+919873884431,9873884431
+919099914996,9099914996
+918048032823,8048032823
+919855172741,9855172741
+919779886555,9779886555
+914144247010,4144247010
+917397589330,7397589330


In [0]:
# Step 7: Add boolean flags for key capabilities
from pyspark.sql.functions import col, when, array_contains, lower

# Define capability keywords
capability_flags = {
    "has_emergency": ["emergency", "er", "trauma", "accident"],
    "has_icu": ["icu", "intensive care", "critical care"],
    "has_obg": ["obg", "obstetrics", "gynecology", "maternity", "gynae"],
    "has_surgery": ["surgery", "operation", "surgical"],
    "has_pharmacy": ["pharmacy", "medical store", "dispensary"],
    "has_lab": ["laboratory", "lab", "pathology", "diagnostic"],
    "has_xray": ["x-ray", "xray", "radiology", "imaging"],
    "has_ambulance": ["ambulance", "emergency transport"]
}

# Check which columns contain capability info (actual column names)
capability_columns = []
for col_name in ["specialties", "procedure", "capability", "equipment", "description"]:
    if col_name in df.columns:
        capability_columns.append(col_name)

print(f"Checking capabilities in columns: {capability_columns}")

# Create boolean flags
for flag_name, keywords in capability_flags.items():
    # Build condition checking all capability columns
    condition = None
    
    for col_name in capability_columns:
        col_type = [f.dataType for f in df.schema.fields if f.name == col_name][0]
        
        if "array" in str(col_type).lower():
            # For array columns, check if any keyword exists in array
            for keyword in keywords:
                keyword_condition = array_contains(col(col_name), keyword)
                condition = keyword_condition if condition is None else (condition | keyword_condition)
        else:
            # For string columns, check if keyword appears in text
            for keyword in keywords:
                keyword_condition = lower(col(col_name)).contains(keyword)
                condition = keyword_condition if condition is None else (condition | keyword_condition)
    
    # Add the boolean column
    if condition is not None:
        df = df.withColumn(flag_name, when(condition, True).otherwise(False))
        print(f"✓ Added {flag_name}")

# Show summary of flags
flag_cols = list(capability_flags.keys())
if all(col_name in df.columns for col_name in flag_cols):
    display(df.select(flag_cols).limit(10))

Checking capabilities in columns: ['specialties', 'procedure', 'capability', 'equipment', 'description']
✓ Added has_emergency
✓ Added has_icu
✓ Added has_obg
✓ Added has_surgery
✓ Added has_pharmacy
✓ Added has_lab
✓ Added has_xray
✓ Added has_ambulance


has_emergency,has_icu,has_obg,has_surgery,has_pharmacy,has_lab,has_xray,has_ambulance
true,false,false,false,false,false,false,false
true,false,false,false,false,false,false,false
true,false,false,false,false,false,false,false
false,false,false,false,false,false,false,false
false,false,false,false,false,false,false,false
false,false,false,false,false,false,false,false
false,false,false,false,false,false,false,false
true,false,false,false,false,false,false,false
false,false,false,false,false,false,false,false
true,false,false,false,false,false,false,false


In [0]:
# Step 8: Build full_text_blob - merge description + capabilities + procedures + equipment
from pyspark.sql.functions import concat_ws, col, coalesce, lit, array_join

# Columns to merge into full text (actual column names)
text_columns = []
for col_name in ["name", "description", "facilityTypeId", "specialties", "procedure", "capability", "equipment"]:
    if col_name in df.columns:
        text_columns.append(col_name)

print(f"Building full_text_blob from: {text_columns}")

# Convert arrays to strings and coalesce
text_parts = []
for col_name in text_columns:
    col_type = [f.dataType for f in df.schema.fields if f.name == col_name][0]
    
    if "array" in str(col_type).lower():
        # Join array elements with commas
        text_parts.append(coalesce(array_join(col(col_name), ", "), lit("")))
    else:
        # Use string as-is
        text_parts.append(coalesce(col(col_name), lit("")))

# Concatenate all parts with spaces
df = df.withColumn(
    "full_text_blob",
    concat_ws(" ", *text_parts)
)

print("✓ Created full_text_blob for vector search")
display(df.select("name", "full_text_blob").limit(3))

Building full_text_blob from: ['name', 'description', 'facilityTypeId', 'specialties', 'procedure', 'capability', 'equipment']
✓ Created full_text_blob for vector search


name,full_text_blob
1000 Smiles Dental Clinic,"1000 Smiles Dental Clinic Dental clinic offering RCT (Root Canal) and Laser Dentistry in Amberpet, Hyderabad. clinic familyMedicine, periodontics, endodontics, dentistry, aestheticDentistry Performs root canal therapy (RCT), Provides laser dentistry services Has been in operation for 10 years, Dental clinic"
108 Eye And Heath Centre,"108 Eye And Heath Centre 108 Eye And Health Centre provides you the best range of treatments, treatment services, pharmaceutical injection, veterinary drugs & our services with effective & timely delivery. clinic pediatricsAndStrabismusOphthalmology, ophthalmology, refractiveSurgeryOphthalmology, familyMedicine, eyeTraumaAndEmergencyEyeCare, cataractAndAnteriorSegmentSurgery, uveitisOphthalmology, retinaAndVitreoretinalOphthalmology, glaucomaOphthalmology, neuroOphthalmology Treats Squint, Glaucoma management, Offers Paediatric Ophthalmology, Performs cataract surgery, General eye check-up, Manages Uveitis, Cataract surgery, LASIK, IPCL, Pre-LASIK tests, Provides Basic Eye Check Up, Offers Retina Treatment Services, Offers Glaucoma Treatment, Offers Neuro Ophthalmology, ICL, Post-LASIK care, Squint correction, Retina treatment, Pediatric ophthalmology services Neuro Ophthalmology services offered, Outpatient ophthalmology clinic, Has 1 ophthalmologist/eye surgeon on staff, Dr. B K Varma is an ophthalmologist/eye surgeon with 17 years of experience, Retina Treatment Services offered, Paediatric Ophthalmology services offered, Employs experienced ophthalmologists, Uses latest equipment and world-standard technologies in eye care, Provides comprehensive eye care services under one roof"
14 Stars Dental,"14 Stars Dental At 14 Stars Dental, we are dedicated to our patients in providing superior dental care at affordable prices. Our goal is to help our patients look and feel better. dentist dentistry, generalDentistry General dentistry services"


In [0]:
# Step 9: Add stable facility_id using MD5 hash
from pyspark.sql.functions import md5, concat_ws, col, coalesce, lit

# Use name + address + phone to create unique ID (actual column names)
id_columns = []
for col_name in ["name", "address_line1", "address_city", "address_stateOrRegion", "clean_phone"]:
    if col_name in df.columns:
        id_columns.append(coalesce(col(col_name), lit("")))

if id_columns:
    df = df.withColumn(
        "facility_id",
        md5(concat_ws("|", *id_columns))
    )
    print("✓ Added stable facility_id (MD5 hash)")
    display(df.select("facility_id", "name", "address_city").limit(10))
else:
    print("⚠ Could not create facility_id - missing required columns")

✓ Added stable facility_id (MD5 hash)


facility_id,name,address_city
2cdbde91d9513d4d8106b1e8ce47ad26,1000 Smiles Dental Clinic,Hyderabad
2d599752ffb94b9895fca71b25f3312c,108 Eye And Heath Centre,Noida
48716c2c3f3671ab715e8fcbe3da5ff1,14 Stars Dental,Aurangabad
cd3522a52926bfa98e0d0678e695836b,24x7 Family Clinix,Ghaziabad
a4008ec17ac653adfea62cd778548358,3 E EYE CARE,Ahmedabad
17a80319913142a1ff869f13c31c4af9,30 + Ayurved Clinic,Pune
a0c70ef1007af7d6ff61796a1abccf25,32 Crowns Dental Clinic,Dinanagar
1cdeb0b1dbe206f89d0e460bc8216a14,32 Dental Care,Amritsar
fa5ca02f2fbcae89a08ebe3c3a036192,32 Dental Care Multispeciality Dental Clinic,Chidambaram
6c545950d9678f3df603362ed488a366,32 Dental Focus,Tiruppur


In [0]:
# Step 10: Write clean_facilities to Unity Catalog
from pyspark.sql.functions import current_timestamp

# Add metadata columns
df_final = df.withColumn("cleaned_at", current_timestamp())

# Write to Unity Catalog
target_table = "workspace.default.clean_facilities"

df_final.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(target_table)

print(f"✅ SUCCESS: Wrote {df_final.count()} rows to {target_table}")
print(f"\n📊 Final Schema:")
df_final.printSchema()

print(f"\n📋 Sample Data:")
display(df_final.limit(10))

✅ SUCCESS: Wrote 10000 rows to workspace.default.clean_facilities

📊 Final Schema:
root
 |-- name: string (nullable = true)
 |-- phone_numbers: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- officialPhone: string (nullable = true)
 |-- email: string (nullable = true)
 |-- websites: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- officialWebsite: string (nullable = true)
 |-- yearEstablished: string (nullable = true)
 |-- facebookLink: string (nullable = true)
 |-- twitterLink: string (nullable = true)
 |-- linkedinLink: string (nullable = true)
 |-- instagramLink: string (nullable = true)
 |-- address_line1: string (nullable = true)
 |-- address_line2: string (nullable = true)
 |-- address_line3: string (nullable = true)
 |-- address_city: string (nullable = true)
 |-- address_stateOrRegion: string (nullable = true)
 |-- address_zipOrPostcode: string (nullable = true)
 |-- address_country: string (nullable = true)
 |-- address_

name,phone_numbers,officialPhone,email,websites,officialWebsite,yearEstablished,facebookLink,twitterLink,linkedinLink,instagramLink,address_line1,address_line2,address_line3,address_city,address_stateOrRegion,address_zipOrPostcode,address_country,address_countryCode,facilityTypeId,operatorTypeId,affiliationTypeIds,description,numberDoctors,capacity,specialties,procedure,equipment,capability,recency_of_page_update,distinct_social_media_presence_count,affiliated_staff_presence,custom_logo_presence,number_of_facts_about_the_organization,post_metrics_most_recent_social_media_post_date,post_metrics_post_count,engagement_metrics_n_followers,engagement_metrics_n_likes,engagement_metrics_n_engagements,latitude,longitude,clean_phone,has_emergency,has_icu,has_obg,has_surgery,has_pharmacy,has_lab,has_xray,has_ambulance,full_text_blob,facility_id,cleaned_at
1000 Smiles Dental Clinic,List(+919392803399),+919392803399,1000smilesdentalclinic@gmail.com,"List(https://www.facebook.com/249768584883702, https://www.justdial.com/Hyderabad/1000-Smiles-Dental-Clinic-Opposie-Hdfc-Bank-Amberpet/040PXX40-XX40-160227012741-N9P1_BZDET)",null,null,https://www.facebook.com/249768584883702,null,null,null,"Plot No 2-2-1075/15/1/5, Shivam Road","Opposite Hdfc Bank, Tilak Nagar","Near Old Mla Quarters, Hyderguda",Hyderabad,Telangana,500013.0,India,IN,clinic,null,null,"Dental clinic offering RCT (Root Canal) and Laser Dentistry in Amberpet, Hyderabad.",null,null,"List(familyMedicine, periodontics, endodontics, dentistry, aestheticDentistry)","List(Performs root canal therapy (RCT), Provides laser dentistry services)",List(),"List(Has been in operation for 10 years, Dental clinic)",null,null,false,false,null,null,null,null,null,null,17.3977394104003,78.482681274414,9392803399,true,false,false,false,false,false,false,false,"1000 Smiles Dental Clinic Dental clinic offering RCT (Root Canal) and Laser Dentistry in Amberpet, Hyderabad. clinic familyMedicine, periodontics, endodontics, dentistry, aestheticDentistry Performs root canal therapy (RCT), Provides laser dentistry services Has been in operation for 10 years, Dental clinic",2cdbde91d9513d4d8106b1e8ce47ad26,2026-04-25T18:51:53.161Z
108 Eye And Heath Centre,List(+919015821616),+919015821616,108healthcentre@gmail.com,"List(https://www.indiamart.com/108-eye-and-health-centre/, https://108-eye-health.localo.site/, 108eye.com, https://www.justdial.com/Noida/108-Eye-And-Heath-Centre-Near-State-Bank-Of-India-D-Block-Sector-50-Noida-Sector-50/011PXX11-XX11-180216202406-Z2S8_BZDET, https://www.facebook.com/108eyeandhealthcentre/, https://108eye.com/, https://www.practo.com/noida/clinics/eye-clinics, https://www.facebook.com/252399348484757)",108eye.com,null,https://www.facebook.com/252399348484757,null,null,null,House Number 108,"Near State Bank Of India D Block Sector 50, E Block",null,Noida,Uttar Pradesh,201307.0,India,IN,clinic,private,null,"108 Eye And Health Centre provides you the best range of treatments, treatment services, pharmaceutical injection, veterinary drugs & our services with effective & timely delivery.",null,null,"List(pediatricsAndStrabismusOphthalmology, ophthalmology, refractiveSurgeryOphthalmology, familyMedicine, eyeTraumaAndEmergencyEyeCare, cataractAndAnteriorSegmentSurgery, uveitisOphthalmology, retinaAndVitreoretinalOphthalmology, glaucomaOphthalmology, neuroOphthalmology)","List(Treats Squint, Glaucoma management, Offers Paediatric Ophthalmology, Performs cataract surgery, General eye check-up, Manages Uveitis, Cataract surgery, LASIK, IPCL, Pre-LASIK tests, Provides Basic Eye Check Up, Offers Retina Treatment Services, Offers Glaucoma Treatment, Offers Neuro Ophthalmology, ICL, Post-LASIK care, Squint correction, Retina treatment, Pediatric ophthalmology services)",List(),"List(Neuro Ophthalmology services offered, Outpatient ophthalmology clinic, Has 1 ophthalmologist/eye surgeon on staff, Dr. B K Varma is an ophthalmologist/eye surgeon with 17 years of experience, Retina Treatment Servi

## ✅ Data Cleaning Complete

The clean healthcare facilities data is now available at:
**`workspace.default.clean_facilities`**

### What's in the clean table:
- ✓ No string "null" or "[]" values
- ✓ State field properly mapped
- ✓ JSON arrays parsed (specialties, services, etc.)
- ✓ "farmacy" → "pharmacy" typo fixed
- ✓ Numeric columns cast to integers
- ✓ Clean 10-digit phone numbers
- ✓ Boolean flags: has_emergency, has_icu, has_obg, etc.
- ✓ full_text_blob for vector search embeddings
- ✓ Stable facility_id (MD5 hash)

